In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "data" / "household_energy_consumption_enriched.csv").exists()
    ),
    Path.cwd(),
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data import FEATURE_COLUMNS, HEATING_TYPE_FEATURES

df = pd.read_csv(PROJECT_ROOT / "data" / "household_energy_consumption_enriched.csv")

df.head()

In [ ]:
df_model = df.copy()

df_model["Date"] = pd.to_datetime(df_model["Date"])
df_model["Has_AC"] = df_model["Has_AC"].astype(str).str.strip().str.title()
df_model["heating_type"] = df_model["heating_type"].astype(str).str.strip().str.title()

df_model["surface_m2"] = pd.to_numeric(df_model["surface_m2"], errors="coerce")
df_model["hours_at_home"] = pd.to_numeric(df_model["hours_at_home"], errors="coerce")
df_model["Has_AC_Binary"] = df_model["Has_AC"].map({
    "Yes": 1,
    "No": 0,
})

for heating_type, feature_name in HEATING_TYPE_FEATURES.items():
    df_model[feature_name] = df_model["heating_type"].eq(heating_type).astype(int)

df_model["day"] = df_model["Date"].dt.day
df_model["day_of_week"] = df_model["Date"].dt.dayofweek

df_model["temperature_x_ac"] = df_model["Avg_Temperature_C"] * df_model["Has_AC_Binary"]
df_model["surface_per_person"] = df_model["surface_m2"] / df_model["Household_Size"].replace(0, pd.NA)

df_model.head()

? partir des r?sultats de l?EDA, plusieurs nouvelles colonnes sont cr??es pour aider le mod?le.

`Has_AC_Binary` transforme la variable `Has_AC` en valeur num?rique : `Yes` devient 1 et `No` devient 0. Cette colonne est utile car l?EDA a montr? que les foyers avec climatisation consomment plus.

`surface_m2`, `heating_type` et `hours_at_home` enrichissent le profil du logement et les habitudes de pr?sence. Le type de chauffage est encod? en variables indicatrices pour rester compatible avec les mod?les scikit-learn.

`temperature_x_ac` combine la temp?rature moyenne et la pr?sence de climatisation.

In [ ]:
features = FEATURE_COLUMNS

X = df_model[features]
y = df_model["Energy_Consumption_kWh"]

X.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
linear_model.fit(X_train, y_train)

In [ ]:
y_pred = linear_model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("R² :", r2)

Les m?triques affich?es ci-dessus permettent de v?rifier l'apport du dataset enrichi. Le mod?le utilise d?sormais la taille du foyer, la temp?rature, la climatisation, la surface du logement, le temps pass? ? domicile, les interactions existantes et l'encodage du type de chauffage.